# Local LLM Self-Host Sandbox

> 💡 **How to Run:** This repository provides the notebook source file (`.ipynb`). To test it with free GPU compute, download the notebook from this repository and import it into your own **[Kaggle](https://www.kaggle.com/)** account (`File` → `Import Notebook`), then enable a GPU accelerator in the session settings.

---

## ⚠️ Disclaimer & Usage Policy

> **Please read carefully before running this notebook.**

### 1. Purpose of Use
This notebook is designed to help developers and researchers **quickly host and test local LLMs (GGUF models) utilizing the free compute resources provided by Kaggle**. It is intended strictly for rapid prototyping, benchmarking, and experimental evaluation.

### 2. Prohibited Activities & Resource Abuse
In accordance with **Kaggle's Terms of Service and Acceptable Use Policy**, you must **NOT** use this notebook to:
* **Host persistent production APIs** or long-running backend services.
* **Serve commercial workloads**, public-facing applications, or third-party integrations.
* **Offload external development workflows**, such as connecting local IDEs, AI coding assistants (e.g., Cline, Continue, Cursor), or automated scraping/agent pipelines to bypass compute limits.
* **Abuse or circumvent GPU/CPU quota restrictions** allocated by the platform.

### 3. Limitation of Liability
* **No Warranty:** This script is provided *"as is"*, without warranty of any kind, express or implied.
* **Account Responsibility:** The author assumes **no responsibility or liability** for any account suspension, quota revocation, data loss, or policy violations resulting from the misuse of this notebook. **You are solely responsible for ensuring your usage complies with the hosting platform's rules.**

In [ ]:
import os
import re
import time
import socket
import shutil
import secrets
import subprocess
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# =====================================================================
# 🛠️ HELPER & HARDWARE AUTO-DETECTION FUNCTIONS
# =====================================================================

def get_free_port(default_port: int = 8080) -> int:
    """Finds an available port, preferring the default port if free."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", default_port)) != 0:
            return default_port
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        return s.getsockname()[1]

def configure_hardware_acceleration() -> list[str]:
    """
    Automatically detects NVIDIA GPUs and returns the optimal llama.cpp
    acceleration flags (-ngl and --tensor-split) without user intervention.
    """
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            encoding="utf-8",
            stderr=subprocess.DEVNULL
        ).strip()
        
        gpus = [line.strip() for line in output.splitlines() if line.strip()]
        gpu_count = len(gpus)
        
        if gpu_count > 1:
            split_ratio = ",".join(["1"] * gpu_count)
            print(f"🎮 Hardware: {gpu_count}x GPUs detected ({gpus[0]}). Offloading all layers (-ngl 99) with tensor-split {split_ratio}.")
            return ["-ngl", "99", "--tensor-split", split_ratio]
        elif gpu_count == 1:
            print(f"🎮 Hardware: 1x GPU detected ({gpus[0]}). Offloading all layers (-ngl 99).")
            return ["-ngl", "99"]
            
    except Exception:
        pass

    print("⚠️ Hardware: No NVIDIA GPU detected. Falling back to CPU execution (-ngl 0).")
    return ["-ngl", "0"]

def derive_alias(hf_repo: str) -> str:
    """Generates a clean model alias from a Hugging Face repo string."""
    if not hf_repo.strip():
        return "custom-model"
    repo_part = hf_repo.strip().split("/")[-1].split(":")[0]
    clean_name = re.sub(r"[-_]?GGUF$", "", repo_part, flags=re.IGNORECASE)
    return clean_name.lower()

def install_dependencies():
    """Installs llama.cpp CLI and cloudflared if not already present."""
    home = Path.home()
    local_bin = home / ".local" / "bin"
    llama_dir = home / ".llama-app"
    
    os.environ["PATH"] += f"{os.pathsep}{local_bin}{os.pathsep}{llama_dir}"

    if not shutil.which("llama") and not (llama_dir / "llama").exists():
        print("⏳ Installing llama.cpp CLI...")
        subprocess.run(
            "curl -LsSf https://llama.app/install.sh | sh",
            shell=True, check=True
        )

    cf_path = Path("./cloudflared")
    if not cf_path.exists() or cf_path.stat().st_size < 1_000_000:
        print("⏳ Downloading cloudflared binary...")
        subprocess.run(
            "wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared",
            shell=True, check=True
        )

def cleanup_processes():
    """Safely terminates any lingering llama or cloudflared processes."""
    for proc_name in ["llama", "cloudflared"]:
        subprocess.run(["pkill", "-x", proc_name], stderr=subprocess.DEVNULL)
    time.sleep(1)

# =====================================================================
# 🚀 SERVER LAUNCHER
# =====================================================================

def launch_server(hf_repo: str, alias: str, context_size: int, flash_attn: bool):
    install_dependencies()
    cleanup_processes()

    port = get_free_port()
    api_key = f"sk-kaggle-{secrets.token_hex(16)}"
    model_alias = alias.strip() if alias.strip() else derive_alias(hf_repo)

    server_cmd = [
        "llama", "serve",
        "-hf", hf_repo.strip(),
        "--alias", model_alias,
        "--host", "0.0.0.0",
        "--port", str(port),
        "-c", str(context_size),
        "--api-key", api_key
    ]

    if flash_attn:
        server_cmd.extend(["-fa", "on"])

    # Automatically inject optimal GPU layer offloading & tensor splitting
    server_cmd.extend(configure_hardware_acceleration())

    print(f"🚀 Starting model [{hf_repo}] on port {port}...")
    llama_log_path = Path("llama_server.log")
    cf_log_path = Path("cloudflared.log")

    with open(llama_log_path, "w") as server_log, open(cf_log_path, "w") as cf_log:
        server_proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)

        print("⏳ Loading model into memory (this may take a few minutes for downloads)...")
        server_ready = False
        for i in range(300):
            time.sleep(2)

            if server_proc.poll() is not None:
                print("❌ Llama server exited unexpectedly! Log tail:")
                print(llama_log_path.read_text()[-1500:])
                return

            log_text = llama_log_path.read_text()
            if i % 5 == 0 and log_text.strip():
                last_line = log_text.strip().splitlines()[-1]
                print(f"   [Progress]: {last_line[:95]}")

            if "listening on" in log_text.lower():
                server_ready = True
                break

        if not server_ready:
            print("❌ Timed out waiting for llama server to start.")
            server_proc.terminate()
            return

        print("🌐 Establishing Cloudflare Tunnel...")
        cf_proc = subprocess.Popen(
            ["./cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
            stdout=cf_log,
            stderr=subprocess.STDOUT
        )

        public_url = None
        for _ in range(30):
            time.sleep(1)
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_log_path.read_text())
            if match:
                public_url = match.group(0)
                break

        print("\n" + "=" * 65)
        if server_ready and public_url:
            print("✅ SYSTEM READY!")
            print(f"👉 Base URL : {public_url}/v1")
            print(f"👉 API Key  : {api_key}")
            print(f"👉 Model    : {model_alias}")
        else:
            print("⚠️ Startup issue detected. Tail of cloudflared.log:")
            print(cf_log_path.read_text()[-1000:])
        print("=" * 65)
        print("💡 Keep this cell running (or interrupt kernel to stop the server).")

        try:
            while True:
                time.sleep(60)
        except KeyboardInterrupt:
            server_proc.terminate()
            cf_proc.terminate()
            print("\n🛑 Server and tunnel stopped.")

# =====================================================================
# 🎛️ KAGGLE INTERACTIVE WIDGET UI
# =====================================================================

DEFAULT_REPO = "bartowski/Qwen_Qwen3.5-9B-GGUF:Q4_K_M"
style = {"description_width": "140px"}
layout = widgets.Layout(width="560px")

w_repo = widgets.Text(
    value=DEFAULT_REPO,
    placeholder="e.g., bartowski/Llama-3.2-3B-Instruct-GGUF:Q4_K_M",
    description="HF Repo (:Quant):",
    style=style,
    layout=layout
)

w_alias = widgets.Text(
    value="",
    placeholder=derive_alias(DEFAULT_REPO),
    description="Model Alias:",
    style=style,
    layout=layout
)

# Live-update Model Alias placeholder whenever HF Repo changes
def on_repo_change(change):
    w_alias.placeholder = derive_alias(change["new"])

w_repo.observe(on_repo_change, names="value")

w_ctx = widgets.Dropdown(
    options=[("4096", 4096), ("8192", 8192), ("16384", 16384), ("32768", 32768)],
    value=8192,
    description="Context Size:",
    style=style,
    layout=layout
)

w_fa = widgets.Checkbox(
    value=True,
    description="Enable Flash Attention (-fa on)",
    indent=True,
    style={"description_width": "140px"},
    layout=widgets.Layout(width="450px")
)

btn_start = widgets.Button(
    description="🚀 Launch LLM Server",
    button_style="success",
    layout=widgets.Layout(width="220px", height="38px")
)

output_area = widgets.Output()

def on_start_clicked(b):
    btn_start.disabled = True
    with output_area:
        clear_output()
        if not w_repo.value.strip():
            print("❌ Please enter a valid Hugging Face GGUF repository.")
            btn_start.disabled = False
            return
        try:
            launch_server(
                hf_repo=w_repo.value,
                alias=w_alias.value,
                context_size=w_ctx.value,
                flash_attn=w_fa.value
            )
        finally:
            btn_start.disabled = False

btn_start.on_click(on_start_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>⚙️ Llama.cpp Kaggle Server Configuration</h3>"),
    w_repo,
    w_alias,
    w_ctx,
    w_fa,
    widgets.HTML("<br>"),
    btn_start,
    output_area
])

display(ui)